
# 06 | Financial and Population Data Integration — Silver Layer

## Project
Victorian Local Government Financial Sustainability &
Performance Intelligence (2020–2025)

## Business Objective
Prepare consistent financial and population datasets
for financial sustainability reporting, population-
adjusted analysis and council peer benchmarking.

## Inputs
- Financial data.csv
- Per-capita financial data.csv
- Population.csv
- Council category data.csv
- workspace.default.silver_performance

## Outputs
- workspace.default.silver_financial
- workspace.default.silver_financial_per_capita
- workspace.default.silver_population

## Data Quality Principles
1. Preserve source values for auditability.
2. Standardise council and financial-year identifiers.
3. Validate numeric values without changing valid signs.
4. Identify missing values and duplicate business keys.
5. Reconcile council-year coverage across datasets.
6. Save separate Silver tables at their natural grain.
7. Do not join detailed financial records directly to
   detailed performance indicator records.

## Next Stage
Notebook 07 — Gold Layer Dimensional Modelling


In [0]:

BASE_PATH = "/Volumes/workspace/default/council_raw"

FILES = {
    "financial": f"{BASE_PATH}/financial 2020-25.csv",
    "per_capita": f"{BASE_PATH}/Per-capita financial data.csv",
    "population": f"{BASE_PATH}/Population.csv",
    "categories": f"{BASE_PATH}/Council category data.csv"
}

for name, path in FILES.items():
    print(name, ":", path)


financial : /Volumes/workspace/default/council_raw/financial 2020-25.csv
per_capita : /Volumes/workspace/default/council_raw/Per-capita financial data.csv
population : /Volumes/workspace/default/council_raw/Population.csv
categories : /Volumes/workspace/default/council_raw/Council category data.csv


In [0]:

BASE_PATH = "/Volumes/workspace/default/council_raw"

files = dbutils.fs.ls(BASE_PATH)

for file in files:
    print(file.name)


Council category data.csv
Estimated resident population Local Government Areas Australia.csv
Indicator data.csv
Indicators 2020-21.csv
Indicators 2021-22.csv
Indicators 2022-23.csv
Indicators 2023-24.csv
Indicators 2024-25.csv
Per-capita financial data.csv
Population.csv
combined_2020_2025/
financial 2020-25.csv


In [0]:

raw = {}

for name, path in FILES.items():
    raw[name] = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", True)
        .csv(path)
    )

    print(
        name,
        "| Rows:", raw[name].count(),
        "| Columns:", len(raw[name].columns)
    )

print("PASS: All four datasets loaded")


financial | Rows: 14211 | Columns: 5
per_capita | Rows: 5369 | Columns: 4
population | Rows: 711 | Columns: 5
categories | Rows: 79 | Columns: 2
PASS: All four datasets loaded


In [0]:

for name, df in raw.items():

    print("\n" + "=" * 60)
    print("DATASET:", name.upper())
    print("=" * 60)

    print("Rows:", df.count())
    print("Columns:", df.columns)

    df.printSchema()

    display(df.limit(5))



DATASET: FINANCIAL
Rows: 14211
Columns: ['Council', 'Year', 'Category', 'Sub-Category', 'Value']
root
 |-- Council: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Value: string (nullable = true)



Council,Year,Category,Sub-Category,Value
Alpine Shire Council,2020-21,Current assets,Cash,"$7,286,000"
Alpine Shire Council,2020-21,Current assets,Other assets,"$289,000"
Alpine Shire Council,2020-21,Current assets,Other financial assets,"$30,500,000"
Alpine Shire Council,2020-21,Current assets,Receivables,"$1,640,000"
Alpine Shire Council,2020-21,Current liabilities,Interest bearing liabilities,$0



DATASET: PER_CAPITA
Rows: 5369
Columns: ['Council', 'Year', 'Attribute', 'Value']
root
 |-- Council: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Attribute: string (nullable = true)
 |-- Value: string (nullable = true)



Council,Year,Attribute,Value
Alpine Shire Council,2016-17,Actual cash outflows for PP&E per capita,-669.4888432
Alpine Shire Council,2016-17,Rates and charges per capita,1347.206837
Alpine Shire Council,2016-17,Total expenses per capita,-1983.383447
Alpine Shire Council,2016-17,Total government grants per capita,754.9454028
Alpine Shire Council,2016-17,Total PP&E per capita,16998.33834



DATASET: POPULATION
Rows: 711
Columns: ['Council', 'Year', 'Population*', '_c3', '*Source: Australian Bureau of Statistics.']
root
 |-- Council: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Population*: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- *Source: Australian Bureau of Statistics.: string (nullable = true)



Council,Year,Population*,_c3,*Source: Australian Bureau of Statistics.
Alpine Shire Council,2016-17,12638,null,null
Alpine Shire Council,2017-18,12730,null,null
Alpine Shire Council,2018-19,12812,null,null
Alpine Shire Council,2019-20,12973,null,null
Alpine Shire Council,2020-21,13012,null,null



DATASET: CATEGORIES
Rows: 79
Columns: ['Council', 'Council Category']
root
 |-- Council: string (nullable = true)
 |-- Council Category: string (nullable = true)



Council,Council Category
Banyule City Council,Metropolitan
Bayside City Council,Metropolitan
Boroondara City Council,Metropolitan
Brimbank City Council,Metropolitan
Darebin City Council,Metropolitan


In [0]:

for name, df in raw.items():

    print("\nDATASET:", name.upper())

    missing_counts = df.select([
        F.sum(
            (
                F.col(f"`{column}`").isNull() |
                (
                    F.trim(
                        F.col(f"`{column}`").cast("string")
                    ) == ""
                )
            ).cast("int")
        ).alias(column)
        for column in df.columns
    ])

    display(missing_counts)



DATASET: FINANCIAL


Council,Year,Category,Sub-Category,Value
0,0,0,0,0



DATASET: PER_CAPITA


Council,Year,Attribute,Value
0,0,0,0



DATASET: POPULATION


Council,Year,Population*,_c3,*Source: Australian Bureau of Statistics.
0,0,0,711,711



DATASET: CATEGORIES


Council,Council Category
0,0


In [0]:

from pyspark.sql import functions as F

EXPECTED_YEARS = [
    "2020-21", "2021-22", "2022-23",
    "2023-24", "2024-25"
]

for name in ["financial", "per_capita", "population"]:

    df = raw[name]

    print("\nDATASET:", name.upper())

    display(
        df.groupBy("Year")
        .count()
        .orderBy("Year")
    )

    project_rows = df.filter(
        F.trim(F.col("Year")).isin(EXPECTED_YEARS)
    )

    print("Total source rows:", df.count())
    print("Project-period rows:", project_rows.count())
    print(
        "Outside project period:",
        df.count() - project_rows.count()
    )



DATASET: FINANCIAL


Year,count
2020-21,2915
2021-22,2754
2022-23,2739
2023-24,2918
2024-25,2885


Total source rows: 14211
Project-period rows: 14211
Outside project period: 0

DATASET: PER_CAPITA


Year,count
2016-17,602
2017-18,602
2018-19,595
2019-20,595
2020-21,595
2021-22,595
2022-23,595
2023-24,595
2024-25,595


Total source rows: 5369
Project-period rows: 2975
Outside project period: 2394

DATASET: POPULATION


Year,count
2016-17,79
2017-18,79
2018-19,79
2019-20,79
2020-21,79
2021-22,79
2022-23,79
2023-24,79
2024-25,79


Total source rows: 711
Project-period rows: 395
Outside project period: 316


In [0]:

from pyspark.sql import functions as F

keys = {
    "financial": [
        "Council", "Year", "Category", "Sub-Category"
    ],
    "per_capita": [
        "Council", "Year", "Attribute"
    ],
    "population": [
        "Council", "Year"
    ]
}

for name, columns in keys.items():

    df = raw[name].filter(
        F.trim(F.col("Year")).isin(EXPECTED_YEARS)
    )

    duplicates = (
        df.groupBy(*columns)
        .count()
        .filter(F.col("count") > 1)
    )

    print("\nDATASET:", name.upper())
    print("Project-period rows:", df.count())
    print(
        "Distinct council names:",
        df.select("Council").distinct().count()
    )
    print(
        "Duplicate business-key groups:",
        duplicates.count()
    )

    display(duplicates.limit(10))



DATASET: FINANCIAL
Project-period rows: 14211
Distinct council names: 79
Duplicate business-key groups: 0


Council,Year,Category,Sub-Category,count



DATASET: PER_CAPITA
Project-period rows: 2975
Distinct council names: 85
Duplicate business-key groups: 0


Council,Year,Attribute,count



DATASET: POPULATION
Project-period rows: 395
Distinct council names: 79
Duplicate business-key groups: 0


Council,Year,count


In [0]:

from pyspark.sql import functions as F

# Load validated council reference
reference = raw["categories"]

def council_key(column):
    name = F.lower(F.trim(F.col(column)))

    # Remove terminal "Council" for matching
    name = F.regexp_replace(
        name, r"\s+council$", ""
    )

    return name

reference_names = (
    reference
    .select(
        F.col("Council").alias("ReferenceCouncil"),
        council_key("Council").alias("CouncilKey")
    )
    .distinct()
)

for name in ["financial", "per_capita", "population"]:

    df = raw[name].filter(
        F.trim(F.col("Year")).isin(EXPECTED_YEARS)
    )

    source_names = (
        df.select("Council")
        .distinct()
        .withColumn(
            "CouncilKey",
            council_key("Council")
        )
    )

    unmatched = (
        source_names
        .join(
            reference_names.select("CouncilKey"),
            "CouncilKey",
            "left_anti"
        )
    )

    print("\nDATASET:", name.upper())
    print(
        "Unmatched council names:",
        unmatched.count()
    )

    display(
        unmatched.orderBy("Council").limit(30)
    )



DATASET: FINANCIAL
Unmatched council names: 0


CouncilKey,Council



DATASET: PER_CAPITA
Unmatched council names: 6


CouncilKey,Council
interface average,Interface Average
large shire average,Large Shire Average
metropolitan average,Metropolitan Average
regional average,Regional Average
sector average,Sector Average
small shire average,Small Shire Average



DATASET: POPULATION
Unmatched council names: 0


CouncilKey,Council


In [0]:

from pyspark.sql import functions as F

def council_key(column):
    return F.regexp_replace(
        F.lower(F.trim(F.col(column))),
        r"\s+council$",
        ""
    )

reference_names = (
    raw["categories"]
    .select(
        F.col("Council").alias("ReferenceCouncil"),
        council_key("Council").alias("CouncilKey")
    )
    .distinct()
)

for name in ["financial", "per_capita", "population"]:

    df = raw[name].filter(
        F.trim(F.col("Year")).isin(EXPECTED_YEARS)
    )

    source_names = (
        df.select("Council")
        .distinct()
        .withColumn(
            "CouncilKey",
            council_key("Council")
        )
    )

    unmatched = source_names.join(
        reference_names.select("CouncilKey"),
        "CouncilKey",
        "left_anti"
    )

    print("\nDATASET:", name.upper())
    print("Unmatched names:", unmatched.count())

    display(unmatched.orderBy("Council"))



DATASET: FINANCIAL
Unmatched names: 0


CouncilKey,Council



DATASET: PER_CAPITA
Unmatched names: 6


CouncilKey,Council
interface average,Interface Average
large shire average,Large Shire Average
metropolitan average,Metropolitan Average
regional average,Regional Average
sector average,Sector Average
small shire average,Small Shire Average



DATASET: POPULATION
Unmatched names: 0


CouncilKey,Council


In [0]:

from pyspark.sql import functions as F

EXPECTED_YEARS = [
    "2020-21", "2021-22", "2022-23",
    "2023-24", "2024-25"
]

benchmark_names = [
    "Interface Average",
    "Large Shire Average",
    "Metropolitan Average",
    "Regional Average",
    "Small Shire Average",
    "Sector Average"
]

per_capita_period = raw["per_capita"].filter(
    F.trim(F.col("Year")).isin(EXPECTED_YEARS)
)

per_capita_councils = per_capita_period.filter(
    ~F.trim(F.col("Council")).isin(benchmark_names)
)

per_capita_benchmarks = per_capita_period.filter(
    F.trim(F.col("Council")).isin(benchmark_names)
)

print("Total project-period rows:",
      per_capita_period.count())

print("Council-level rows:",
      per_capita_councils.count())

print("Benchmark rows:",
      per_capita_benchmarks.count())

print("Council identities:",
      per_capita_councils.select("Council").distinct().count())

print("Benchmark labels:",
      per_capita_benchmarks.select("Council").distinct().count())

assert (
    per_capita_councils.count() +
    per_capita_benchmarks.count()
    == per_capita_period.count()
)

display(
    per_capita_benchmarks.groupBy("Council")
    .count()
    .orderBy("Council")
)


Total project-period rows: 2975
Council-level rows: 2765
Benchmark rows: 210
Council identities: 79
Benchmark labels: 6


Council,count
Interface Average,35
Large Shire Average,35
Metropolitan Average,35
Regional Average,35
Sector Average,35
Small Shire Average,35


In [0]:

from pyspark.sql import functions as F

BASE_PATH = "/Volumes/workspace/default/council_raw"

FILES = {
    "financial": f"{BASE_PATH}/financial 2020-25.csv",
    "per_capita": f"{BASE_PATH}/Per-capita financial data.csv",
    "population": f"{BASE_PATH}/Population.csv",
    "categories": f"{BASE_PATH}/Council category data.csv"
}

EXPECTED_YEARS = [
    "2020-21", "2021-22", "2022-23",
    "2023-24", "2024-25"
]

raw = {}

for name, path in FILES.items():
    raw[name] = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", True)
        .csv(path)
    )

    print(name, raw[name].count())


financial 14211
per_capita 5369
population 711
categories 79


In [0]:

from pyspark.sql import functions as F

# Six published benchmark categories
benchmark_names = [
    "Interface Average",
    "Large Shire Average",
    "Metropolitan Average",
    "Regional Average",
    "Small Shire Average",
    "Sector Average"
]

# Filter to the five financial years
per_capita_period = raw["per_capita"].filter(
    F.trim(F.col("Year")).isin(EXPECTED_YEARS)
)

# Individual council records
per_capita_councils = per_capita_period.filter(
    ~F.trim(F.col("Council")).isin(benchmark_names)
)

# Published benchmark records
per_capita_benchmarks = per_capita_period.filter(
    F.trim(F.col("Council")).isin(benchmark_names)
)

print("Total records:", per_capita_period.count())
print("Council records:", per_capita_councils.count())
print("Benchmark records:", per_capita_benchmarks.count())

assert (
    per_capita_councils.count()
    + per_capita_benchmarks.count()
    == per_capita_period.count()
)


Total records: 2975
Council records: 2765
Benchmark records: 210


In [0]:

from pyspark.sql import functions as F

# Standardise council names using validated mappings
def standardise_council(column):
    name = F.trim(F.col(column))

    return (
        F.when(name == "Moreland City", "Merri-bek City")
         .when(
             name == "Colac-Otway Shire",
             "Colac Otway Shire"
         )
         .when(
             name == "Greater Shepparton",
             "Greater Shepparton City"
         )
         .otherwise(name)
    )

# Financial data
financial_clean = (
    raw["financial"]
    .filter(F.trim(F.col("Year")).isin(EXPECTED_YEARS))
    .withColumn("OriginalCouncil", F.col("Council"))
    .withColumn(
        "Council",
        standardise_council("Council")
    )
    .withColumnRenamed("Year", "FinancialYear")
)

# Per-capita council records only
per_capita_clean = (
    per_capita_councils
    .withColumn("OriginalCouncil", F.col("Council"))
    .withColumn(
        "Council",
        standardise_council("Council")
    )
    .withColumnRenamed("Year", "FinancialYear")
)

# Population data
population_clean = (
    raw["population"]
    .filter(F.trim(F.col("Year")).isin(EXPECTED_YEARS))
    .withColumn("OriginalCouncil", F.col("Council"))
    .withColumn(
        "Council",
        standardise_council("Council")
    )
    .withColumnRenamed("Year", "FinancialYear")
)

# Preserve published benchmark records separately
benchmark_clean = (
    per_capita_benchmarks
    .withColumnRenamed("Year", "FinancialYear")
    .withColumnRenamed("Council", "BenchmarkName")
)

datasets = {
    "Financial": financial_clean,
    "Per-capita": per_capita_clean,
    "Population": population_clean
}

for name, df in datasets.items():
    print("\nDATASET:", name)
    print("Records:", df.count())
    print(
        "Standardised councils:",
        df.select("Council").distinct().count()
    )

print("\nBenchmark records:", benchmark_clean.count())



DATASET: Financial
Records: 14211
Standardised councils: 79

DATASET: Per-capita
Records: 2765
Standardised councils: 79

DATASET: Population
Records: 395
Standardised councils: 79

Benchmark records: 210


In [0]:
display(financial_clean.limit(10))
display(per_capita_clean.limit(10))
display(population_clean.limit(10))
display(benchmark_clean.limit(10))

Council,FinancialYear,Category,Sub-Category,Value,OriginalCouncil
Alpine Shire Council,2020-21,Current assets,Cash,"$7,286,000",Alpine Shire Council
Alpine Shire Council,2020-21,Current assets,Other assets,"$289,000",Alpine Shire Council
Alpine Shire Council,2020-21,Current assets,Other financial assets,"$30,500,000",Alpine Shire Council
Alpine Shire Council,2020-21,Current assets,Receivables,"$1,640,000",Alpine Shire Council
Alpine Shire Council,2020-21,Current liabilities,Interest bearing liabilities,$0,Alpine Shire Council
Alpine Shire Council,2020-21,Current liabilities,Lease liabilities,$0,Alpine Shire Council
Alpine Shire Council,2020-21,Current liabilities,Other liabilities,"$6,581,000",Alpine Shire Council
Alpine Shire Council,2020-21,Current liabilities,Provisions,"$2,614,000",Alpine Shire Council
Alpine Shire Council,2020-21,Current liabilities,Trade and other payables,"$4,077,000",Alpine Shire Council
Alpine Shire Council,2020-21,Current liabilities,Trust funds and deposits,"$278,000",Alpine Shire Council


Council,FinancialYear,Attribute,Value,OriginalCouncil
Alpine Shire Council,2020-21,Actual cash outflows for PP&E per capita,-424.6875499,Alpine Shire Council
Alpine Shire Council,2020-21,Rates and charges per capita,1472.649807,Alpine Shire Council
Alpine Shire Council,2020-21,Total expenses per capita,-2361.911414,Alpine Shire Council
Alpine Shire Council,2020-21,Total government grants per capita,848.9908368,Alpine Shire Council
Alpine Shire Council,2020-21,Total PP&E per capita,17063.97035,Alpine Shire Council
Alpine Shire Council,2020-21,Total revenue per capita,2725.040019,Alpine Shire Council
Alpine Shire Council,2020-21,Total user fees and statutory fees per capita,133.5698447,Alpine Shire Council
Alpine Shire Council,2021-22,Actual cash outflows for PP&E per capita,-1191.598439,Alpine Shire Council
Alpine Shire Council,2021-22,Rates and charges per capita,1512.434004,Alpine Shire Council
Alpine Shire Council,2021-22,Total expenses per capita,-2376.769454,Alpine Shire Council


Council,FinancialYear,Population*,_c3,*Source: Australian Bureau of Statistics.,OriginalCouncil
Alpine Shire Council,2020-21,13012,null,null,Alpine Shire Council
Alpine Shire Council,2021-22,13069,null,null,Alpine Shire Council
Alpine Shire Council,2022-23,13175,null,null,Alpine Shire Council
Alpine Shire Council,2023-24,13182,null,null,Alpine Shire Council
Alpine Shire Council,2024-25,13199,null,null,Alpine Shire Council
Ararat Rural City Council,2020-21,11965,null,null,Ararat Rural City Council
Ararat Rural City Council,2021-22,11922,null,null,Ararat Rural City Council
Ararat Rural City Council,2022-23,11741,null,null,Ararat Rural City Council
Ararat Rural City Council,2023-24,11683,null,null,Ararat Rural City Council
Ararat Rural City Council,2024-25,11686,null,null,Ararat Rural City Council


BenchmarkName,FinancialYear,Attribute,Value
Interface Average,2020-21,Actual cash outflows for PP&E per capita,-348.1711006
Interface Average,2020-21,Rates and charges per capita,850.6302597
Interface Average,2020-21,Total expenses per capita,-1225.380947
Interface Average,2020-21,Total government grants per capita,273.4172175
Interface Average,2020-21,Total PP&E per capita,14822.57167
Interface Average,2020-21,Total revenue per capita,1799.491785
Interface Average,2020-21,Total user fees and statutory fees per capita,117.6781354
Interface Average,2021-22,Actual cash outflows for PP&E per capita,-364.4446812
Interface Average,2021-22,Rates and charges per capita,910.3675331
Interface Average,2021-22,Total expenses per capita,-1311.91054


In [0]:

from pyspark.sql import functions as F

numeric_sources = {
    "Financial": financial_clean,
    "Per-capita": per_capita_clean,
    "Benchmarks": benchmark_clean
}

for name, df in numeric_sources.items():

    value_text = F.trim(F.col("Value"))

    invalid = df.filter(
        value_text.isNotNull()
        & (value_text != "")
        & ~value_text.rlike(
            r"^\(?-?\$?[0-9,]+(\.[0-9]+)?\)?$"
        )
    )

    print("\nDATASET:", name)
    print("Total records:", df.count())
    print("Unexpected numeric formats:", invalid.count())

    display(invalid.select("Value").distinct().limit(20))

print("\nPOPULATION")

invalid_population = population_clean.filter(
    F.col("Population*").isNotNull()
    & ~F.trim(F.col("Population*")).rlike(r"^[0-9,]+$")
)

print(
    "Unexpected population formats:",
    invalid_population.count()
)

display(
    invalid_population.select("Population*")
    .distinct()
    .limit(20)
)



DATASET: Financial
Total records: 14211
Unexpected numeric formats: 0


Value



DATASET: Per-capita
Total records: 2765
Unexpected numeric formats: 0


Value



DATASET: Benchmarks
Total records: 210
Unexpected numeric formats: 0


Value



POPULATION
Unexpected population formats: 0


Population*


In [0]:
display(raw["financial"].limit(100))

Council,Year,Category,Sub-Category,Value
Alpine Shire Council,2020-21,Current assets,Cash,"$7,286,000"
Alpine Shire Council,2020-21,Current assets,Other assets,"$289,000"
Alpine Shire Council,2020-21,Current assets,Other financial assets,"$30,500,000"
Alpine Shire Council,2020-21,Current assets,Receivables,"$1,640,000"
Alpine Shire Council,2020-21,Current liabilities,Interest bearing liabilities,$0
Alpine Shire Council,2020-21,Current liabilities,Lease liabilities,$0
Alpine Shire Council,2020-21,Current liabilities,Other liabilities,"$6,581,000"
Alpine Shire Council,2020-21,Current liabilities,Provisions,"$2,614,000"
Alpine Shire Council,2020-21,Current liabilities,Trade and other payables,"$4,077,000"
Alpine Shire Council,2020-21,Current liabilities,Trust funds and deposits,"$278,000"


In [0]:

from pyspark.sql import functions as F

# Financial amounts: currency text to decimal
financial_numeric = (
    financial_clean
    .withColumn("OriginalValue", F.col("Value"))
    .withColumn(
        "Value",
        F.expr("""
            try_cast(
                regexp_replace(trim(Value), '[$,]', '')
                AS DECIMAL(24,4)
            )
        """)
    )
)

# Per-capita financial values
per_capita_numeric = (
    per_capita_clean
    .withColumn("OriginalValue", F.col("Value"))
    .withColumn(
        "Value",
        F.expr("""
            try_cast(
                regexp_replace(trim(Value), '[$,]', '')
                AS DECIMAL(24,8)
            )
        """)
    )
)

# Published benchmark values
benchmark_numeric = (
    benchmark_clean
    .withColumn("OriginalValue", F.col("Value"))
    .withColumn(
        "Value",
        F.expr("""
            try_cast(
                regexp_replace(trim(Value), '[$,]', '')
                AS DECIMAL(24,8)
            )
        """)
    )
)

# Population: text to whole number
population_numeric = (
    population_clean
    .withColumn("OriginalPopulation", F.col("Population*"))
    .withColumn(
        "Population",
        F.expr("""
            try_cast(
                regexp_replace(trim(`Population*`), ',', '')
                AS BIGINT
            )
        """)
    )
    .drop(
        "Population*",
        "_c3",
        "*Source: Australian Bureau of Statistics."
    )
)

# Check results
for name, df in {
    "Financial": financial_numeric,
    "Per-capita": per_capita_numeric,
    "Benchmarks": benchmark_numeric,
    "Population": population_numeric
}.items():
    print("\nDATASET:", name)
    print("Records:", df.count())
    df.printSchema()



DATASET: Financial
Records: 14211
root
 |-- Council: string (nullable = true)
 |-- FinancialYear: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Value: decimal(24,4) (nullable = true)
 |-- OriginalCouncil: string (nullable = true)
 |-- OriginalValue: string (nullable = true)


DATASET: Per-capita
Records: 2765
root
 |-- Council: string (nullable = true)
 |-- FinancialYear: string (nullable = true)
 |-- Attribute: string (nullable = true)
 |-- Value: decimal(24,8) (nullable = true)
 |-- OriginalCouncil: string (nullable = true)
 |-- OriginalValue: string (nullable = true)


DATASET: Benchmarks
Records: 210
root
 |-- BenchmarkName: string (nullable = true)
 |-- FinancialYear: string (nullable = true)
 |-- Attribute: string (nullable = true)
 |-- Value: decimal(24,8) (nullable = true)
 |-- OriginalValue: string (nullable = true)


DATASET: Population
Records: 395
root
 |-- Council: string (nullable = true)
 |-- FinancialY

In [0]:

from pyspark.sql import functions as F

numeric_datasets = {
    "Financial": financial_numeric,
    "Per-capita": per_capita_numeric,
    "Benchmarks": benchmark_numeric
}

for name, df in numeric_datasets.items():

    conversion_failures = df.filter(
        F.col("OriginalValue").isNotNull()
        & (F.trim(F.col("OriginalValue")) != "")
        & F.col("Value").isNull()
    )

    print("\nDATASET:", name)
    print("Total records:", df.count())
    print("Conversion failures:", conversion_failures.count())
    print(
        "Null numeric values:",
        df.filter(F.col("Value").isNull()).count()
    )

    if conversion_failures.count() > 0:
        display(conversion_failures.limit(20))

# Population validation
population_failures = population_numeric.filter(
    F.col("OriginalPopulation").isNotNull()
    & (F.trim(F.col("OriginalPopulation")) != "")
    & F.col("Population").isNull()
)

invalid_population = population_numeric.filter(
    F.col("Population") <= 0
)

print("\nDATASET: Population")
print("Total records:", population_numeric.count())
print("Conversion failures:", population_failures.count())
print(
    "Null population values:",
    population_numeric.filter(
        F.col("Population").isNull()
    ).count()
)
print(
    "Zero or negative population:",
    invalid_population.count()
)

if population_failures.count() > 0:
    display(population_failures.limit(20))

if invalid_population.count() > 0:
    display(invalid_population.limit(20))



DATASET: Financial
Total records: 14211
Conversion failures: 0
Null numeric values: 0

DATASET: Per-capita
Total records: 2765
Conversion failures: 0
Null numeric values: 0

DATASET: Benchmarks
Total records: 210
Conversion failures: 0
Null numeric values: 0

DATASET: Population
Total records: 395
Conversion failures: 0
Null population values: 0
Zero or negative population: 0


In [0]:

from pyspark.sql import functions as F

def council_key(column):
    return F.regexp_replace(
        F.lower(F.trim(F.col(column))),
        r"\s+council$",
        ""
    )

reference_keys = (
    raw["categories"]
    .select(council_key("Council").alias("CouncilKey"))
    .distinct()
)

checks = {
    "Financial": (
        financial_numeric,
        ["Council", "FinancialYear", "Category", "Sub-Category"]
    ),
    "Per-capita": (
        per_capita_numeric,
        ["Council", "FinancialYear", "Attribute"]
    ),
    "Population": (
        population_numeric,
        ["Council", "FinancialYear"]
    )
}

for name, (df, keys) in checks.items():

    source_names = (
        df.select("Council").distinct()
        .withColumn("CouncilKey", council_key("Council"))
    )

    unmatched = source_names.join(
        reference_keys,
        "CouncilKey",
        "left_anti"
    )

    duplicates = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
    )

    print("\nDATASET:", name)
    print("Unmatched council names:", unmatched.count())
    print("Duplicate business-key groups:", duplicates.count())

    if unmatched.count() > 0:
        display(unmatched)

    if duplicates.count() > 0:
        display(duplicates)



DATASET: Financial
Unmatched council names: 0
Duplicate business-key groups: 0

DATASET: Per-capita
Unmatched council names: 0
Duplicate business-key groups: 0

DATASET: Population
Unmatched council names: 0
Duplicate business-key groups: 0


In [0]:

table_names = [
    "workspace.default.silver_financial",
    "workspace.default.silver_financial_per_capita",
    "workspace.default.silver_population",
    "workspace.default.silver_financial_benchmarks"
]

for table_name in table_names:
    print(
        table_name,
        "EXISTS" if spark.catalog.tableExists(table_name)
        else "NOT CREATED"
    )


workspace.default.silver_financial NOT CREATED
workspace.default.silver_financial_per_capita NOT CREATED
workspace.default.silver_population NOT CREATED
workspace.default.silver_financial_benchmarks NOT CREATED


In [0]:

silver_tables = {
    "workspace.default.silver_financial": financial_numeric,
    "workspace.default.silver_financial_per_capita": per_capita_numeric,
    "workspace.default.silver_population": population_numeric,
    "workspace.default.silver_financial_benchmarks": benchmark_numeric
}

for table_name, df in silver_tables.items():

    if spark.catalog.tableExists(table_name):
        print(f"SKIPPED - Already exists: {table_name}")
        continue

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(table_name)
    )

    saved_rows = spark.table(table_name).count()

    print(f"CREATED: {table_name}")
    print(f"Saved rows: {saved_rows}")

    assert saved_rows == df.count()

print("\nSilver table saving process finished.")


CREATED: workspace.default.silver_financial
Saved rows: 14211
CREATED: workspace.default.silver_financial_per_capita
Saved rows: 2765
CREATED: workspace.default.silver_population
Saved rows: 395
CREATED: workspace.default.silver_financial_benchmarks
Saved rows: 210

Silver table saving process finished.
